# playIT audio pipeline (Kokoro-82M)

Produces the Hear It / Say It clips described in `docs/specs/hear-say-refactor.md` §2.2–2.4 and §6.5:

1. Install Kokoro and dependencies
2. Choose a voice from three American English candidates
3. Generate carrier phrases, feedback fragments, and the Table 4 key words
4. Generate the 13 continuous sounds (phoneme input first, key-word extraction as fallback)
5. Add human recordings of the 13 short sounds (optional upload)
6. Run the automated part of Gate 1 and write `manifest.json`
7. Zip everything for download

**Scope.** Short sounds are never synthesized (Table 5). Gate 1 in this notebook is an automated screen. The Praat spectrogram check, the blind listening screen (Gate 2), and the teacher audit (Gate 3) still decide release. Every clip is written with `gate2` and `gate3` set to `"pending"`.

Run on a Colab GPU runtime if you have one (Runtime → Change runtime type). The CPU runtime works but is slower.

## 0. Configuration

In [ ]:
# ---- Change these -----------------------------------------------------------
VOICE = "af_heart"          # chosen voice; set after listening in section 2
SPEED = 1.0                 # speaking rate for carriers, feedback and key words
VOICE_CANDIDATES = ["af_heart", "af_bella", "am_michael"]   # American English (lang_code "a")

OUT = "/content/playit_audio"
SR = 24000                  # Kokoro output rate
PAD_MS = 50                 # §2.3: fixed pad at each end
FADE_MS = 8                 # §2.3: 5-10 ms fade
TARGET_RMS_DBFS = -20.0     # loudness normalization target (RMS)
PEAK_CEILING_DBFS = -1.0    # never exceed this peak after normalization
CONT_TARGET_MS = 800        # Table 4: "about 800 ms"
CONT_BAND_MS = (700, 900)   # AUDIT ASSUMPTION: Table 4 gives no tolerance for "about 800 ms"
CLIP_VERSION = "1"

# Manual key-word cut points (seconds, in the raw key-word audio) that override the
# automatic fallback segmentation for a letter, e.g. {"m": (0.03, 0.12)}.
MANUAL_CUTS = {}

## 1. Install

In [ ]:
!pip install -q "kokoro>=0.9.4" soundfile
!apt-get -qq -y install espeak-ng > /dev/null

In [ ]:
import json, os, shutil, datetime
from importlib.metadata import version

import numpy as np
import soundfile as sf
import librosa
from IPython.display import Audio, display
from kokoro import KPipeline

KOKORO_VERSION = version("kokoro")
MISAKI_VERSION = version("misaki")
TOOL = f"kokoro-82M (kokoro {KOKORO_VERSION}, misaki {MISAKI_VERSION})"
LICENSE = "Apache-2.0 (Kokoro-82M code and weights)"
print(TOOL, "| librosa", librosa.__version__)

pipeline = KPipeline(lang_code="a")   # American English

for sub in ["candidates", "carriers", "feedback", "keywords", "phonemes", "review"]:
    os.makedirs(os.path.join(OUT, sub), exist_ok=True)

## 2. Voice candidates

Listen to the same four sentences in each voice, then set `VOICE` in section 0 and re-run that cell. The sample contains no letter names on purpose.

In [ ]:
SAMPLE = ("Listen! Here is a mouse, a fan, and a big red ball. "
          "Say it with me! "
          "Yes, you did it all by yourself. "
          "Let's try the next one together.")

def to_np(a):
    """Kokoro returns torch tensors; convert to a float32 numpy array."""
    if hasattr(a, "detach"):
        a = a.detach().cpu().numpy()
    return np.asarray(a, dtype=np.float32)

def synth(text, voice, speed=1.0):
    """Normal text-to-speech; joins Kokoro's chunks into one float32 array."""
    parts = [to_np(r.audio) for r in pipeline(text, voice=voice, speed=speed) if r.audio is not None]
    return np.concatenate(parts) if parts else np.zeros(0, np.float32)

for v in VOICE_CANDIDATES:
    y = synth(SAMPLE, v)
    sf.write(os.path.join(OUT, "candidates", f"voice_{v}.wav"), y, SR, subtype="PCM_16")
    print(v)
    display(Audio(y, rate=SR))

## 3. Shared helpers

Every clip goes through `finish_clip`, which applies the §2.3 editing standard: trim silence, add 50 ms pads, fade in and out, normalize loudness, and write a mono 16-bit WAV.

In [ ]:
FRAME = int(0.010 * SR)    # 10 ms analysis frames
HOP = int(0.005 * SR)      # 5 ms hop

def frame_rms_db(y):
    if len(y) < FRAME:
        return np.array([-120.0])
    fr = librosa.util.frame(y, frame_length=FRAME, hop_length=HOP)
    return 20 * np.log10(np.sqrt(np.mean(fr ** 2, axis=0)) + 1e-9)

def active_bounds(y, floor_db=-45.0, rel_db=-35.0):
    """First/last sample whose 10 ms frame is above max(floor, peak + rel)."""
    db = frame_rms_db(y)
    thr = max(floor_db, db.max() + rel_db)
    idx = np.where(db > thr)[0]
    if len(idx) == 0:
        return 0, 0
    return idx[0] * HOP, min(len(y), idx[-1] * HOP + FRAME)

def trim_silence(y):
    a, b = active_bounds(y)
    return y[a:b]

def measure_sound_ms(y):
    """Length of the sound itself, i.e. the active span without pads (Table 4 basis)."""
    a, b = active_bounds(y)
    return 1000.0 * (b - a) / SR

def fade(y, ms=FADE_MS):
    n = min(int(ms / 1000 * SR), len(y) // 2)
    if n > 0:
        ramp = np.linspace(0.0, 1.0, n, dtype=np.float32)
        y = y.copy()
        y[:n] *= ramp
        y[-n:] *= ramp[::-1]
    return y

def normalize(y):
    rms = np.sqrt(np.mean(y ** 2)) + 1e-9
    y = y * (10 ** (TARGET_RMS_DBFS / 20) / rms)
    peak = np.max(np.abs(y)) + 1e-9
    ceiling = 10 ** (PEAK_CEILING_DBFS / 20)
    if peak > ceiling:
        y = y * (ceiling / peak)
    return y

def finish_clip(y, path, already_trimmed=False):
    """§2.3 editing standard; returns (duration_ms, sound_ms)."""
    y = np.asarray(y, dtype=np.float32)
    if y.ndim > 1:
        y = y.mean(axis=1)
    if not already_trimmed:
        y = trim_silence(y)
    y = normalize(fade(y))
    pad = np.zeros(int(PAD_MS / 1000 * SR), np.float32)
    y = np.concatenate([pad, y, pad])
    sf.write(path, y, SR, subtype="PCM_16")
    return round(1000.0 * len(y) / SR), round(measure_sound_ms(y))

MANIFEST = {}   # clipId -> entry (SDD §6.5 fields)

def record(clip_id, type_, path, text=None, letter=None, ipa=None, source="kokoro-tts",
           settings=None, voice=None, tool=None, license_=None):
    dur, snd = None, None
    if path and os.path.exists(path):
        y, _ = sf.read(path, dtype="float32")
        dur, snd = round(1000.0 * len(y) / SR), round(measure_sound_ms(y))
    MANIFEST[clip_id] = {
        "clipId": clip_id, "type": type_, "letter": letter, "ipa": ipa, "text": text,
        "file": os.path.relpath(path, OUT) if path else None,
        "durationMs": dur, "soundMs": snd, "source": source,
        "tool": tool or TOOL, "voice": voice if voice is not None else VOICE,
        "settings": settings or {"speed": SPEED}, "license": license_ or LICENSE,
        "gate1": None, "gate2": "pending", "gate3": "pending",
        "version": CLIP_VERSION, "releaseDate": None,
    }

## 4. Carrier phrases, feedback fragments, and key words

Every line that contains a pure sound (for example "Yes! /m/, lips together") is generated as **fragments**, and the app's AudioComposer joins them with the `ph_` clip (§2.3, Table 5). Asking the voice for the whole line would make it say the letter name. `COMPOSE` records each full line as its sequence of clip IDs and is written to the manifest.

In [ ]:
CARRIERS = {                                   # Table 3
    "car_listen": "Listen!",
    "car_this_letter_says": "This letter says...",
    "car_say_it_with_me": "Say it with me!",
}

FEEDBACK = {                                   # Tables 2 and 7, split around /sound/
    "fb_yes": "Yes!",
    "fb_your_turn": "Your turn.",
    "fb_listen": "Listen:",
    "fb_letter_name": "That's the letter's name.",
    "fb_its_sound_is": "Its sound is",
    "fb_almost_just": "Almost! Just",
    "fb_no_ah": "no 'ah.'",
    "fb_lips_together": "lips together.",       # praise cue for m (Table 2)
    "fb_top_teeth_on_lip": "Top teeth on your lip.",  # substitution cue for f (Section 3.2)
}

# {L} is replaced by ph_<letter> for each letter at runtime.
COMPOSE = {
    "praise_specific":   ["fb_yes", "ph_{L}", "cue_{L}"],       # "Yes! /m/, lips together"
    "remodel_silence":   ["fb_listen", "ph_{L}", "fb_your_turn"],# "Listen: /m/. Your turn."
    "corr_letter_name":  ["fb_letter_name", "fb_its_sound_is", "ph_{L}"],
    "corr_added_vowel":  ["fb_almost_just", "ph_{L}", "fb_no_ah"],
    "corr_substitution": ["fb_listen", "ph_{L}", "subcue_{L}"], # "Listen: /f/. Top teeth on your lip."
}
ARTICULATION_CUES = {"m": "fb_lips_together"}       # cue_{L}; only m is written in the spec
SUBSTITUTION_CUES = {"f": "fb_top_teeth_on_lip"}    # subcue_{L}; only f is written in the spec

TABLE4 = {  # letter: (ipa, type, key word, short-sound limit in ms or None)
    "a": ("æ", "continuous", "apple", None),   "b": ("b", "short", "ball", 250),
    "c": ("k", "short", "cat", 250),           "d": ("d", "short", "dog", 250),
    "e": ("ɛ", "continuous", "egg", None),     "f": ("f", "continuous", "fan", None),
    "g": ("g", "short", "goat", 250),          "h": ("h", "short", "hat", 300),
    "i": ("ɪ", "continuous", "insect", None),  "j": ("dʒ", "short", "jam", 250),
    "k": ("k", "short", "kite", 250),          "l": ("l", "continuous", "leaf", None),
    "m": ("m", "continuous", "mouse", None),   "n": ("n", "continuous", "nest", None),
    "o": ("ɑ", "continuous", "octopus", None), "p": ("p", "short", "pig", 250),
    "q": ("kw", "short", "queen", 300),        "r": ("ɹ", "continuous", "rabbit", None),
    "s": ("s", "continuous", "sun", None),     "t": ("t", "short", "top", 250),
    "u": ("ʌ", "continuous", "umbrella", None),"v": ("v", "continuous", "van", None),
    "w": ("w", "short", "web", 300),           "x": ("ks", "short", "box", 350),
    "y": ("j", "short", "yo-yo", 300),         "z": ("z", "continuous", "zebra", None),
}

def gen_line(clip_id, text, folder, type_, letter=None):
    path = os.path.join(OUT, folder, f"{clip_id}.wav")
    finish_clip(synth(text, VOICE, SPEED), path)
    record(clip_id, type_, path, text=text, letter=letter)

for cid, text in CARRIERS.items():
    gen_line(cid, text, "carriers", "carrier")
for cid, text in FEEDBACK.items():
    gen_line(cid, text, "feedback", "feedback")
for letter, (ipa, kind, word, _) in TABLE4.items():
    gen_line("kw_" + word.replace("-", ""), word, "keywords", "keyword", letter=letter)

missing_cues = [l for l in TABLE4 if l not in ARTICULATION_CUES]
print(f"{len(MANIFEST)} lines generated.")
print("TODO (teachers): articulation-cue praise not yet written in the spec for:", " ".join(missing_cues))
display(Audio(os.path.join(OUT, "carriers", "car_listen.wav")))

## 5. The 13 continuous sounds

**Recommended method (Table 5):** give Kokoro phoneme input for a held sound. Phoneme input spells the sound, not the letter, so the voice is never asked for a letter alone. A small grid of repeat counts and speeds is tried. For each take, the steadiest window of about 800 ms (lowest frame-to-frame spectral flux) is kept, and the take is scored on steadiness and on how close its steady part comes to 800 ms.

**Fallback:** if no take passes the Gate 1 length check, the key word is synthesized and its initial target segment is cut out (onset to the first spectral-flux peak at least 40 ms later, or `MANUAL_CUTS`). The middle 60% is kept and time-stretched to 800 ms while keeping pitch.

Every take and the key-word source audio are saved to `review/` for listeners.

In [ ]:
CONTINUOUS = [l for l, v in TABLE4.items() if v[1] == "continuous"]
REPEATS = [1, 3, 6]
SPEEDS = [0.6, 0.8, 1.0]

def synth_phonemes(ps, voice, speed):
    """Phoneme input. Prefer generate_from_tokens(str); fall back to misaki's inline [text](/phonemes/)."""
    try:
        parts = [to_np(r.audio)
                 for r in pipeline.generate_from_tokens(ps, voice=voice, speed=speed)
                 if r.audio is not None]
        if parts:
            return np.concatenate(parts), "generate_from_tokens"
    except Exception as e:  # older kokoro or rejected token string
        print("  generate_from_tokens failed:", type(e).__name__, e)
    return synth(f"[sound](/{ps}/)", voice, speed), "inline-markdown"

def spectral_flux(y):
    S = np.abs(librosa.stft(y, n_fft=512, hop_length=HOP))
    S = S / (S.sum(axis=0, keepdims=True) + 1e-9)
    return np.r_[0.0, np.sqrt((np.diff(S, axis=1) ** 2).sum(axis=0))]

def steadiest_window(y, target_ms=CONT_TARGET_MS):
    """Returns (segment, mean flux). Window is the target length, or the whole clip if shorter."""
    y = trim_silence(y)
    n = int(target_ms / 1000 * SR)
    if len(y) <= n:
        return y, float(spectral_flux(y).mean()) if len(y) > 512 else 1.0
    flux = spectral_flux(y)
    w = max(1, n // HOP)
    csum = np.r_[0.0, np.cumsum(flux)]
    means = (csum[w:] - csum[:-w]) / w
    start = int(np.argmin(means[: max(1, len(flux) - w)]))
    a = start * HOP
    return y[a:a + n], float(means[start])

def in_band(ms):
    return CONT_BAND_MS[0] <= ms <= CONT_BAND_MS[1]

def extract_from_keyword(letter, word):
    """Fallback: cut the word-initial target sound and stretch it to 800 ms."""
    y = synth(word, VOICE, SPEED)
    sf.write(os.path.join(OUT, "review", f"kwsrc_{letter}_{word}.wav"), y, SR, subtype="PCM_16")
    if letter in MANUAL_CUTS:
        s, e = MANUAL_CUTS[letter]
        seg, how = y[int(s * SR):int(e * SR)], f"manual {s:.3f}-{e:.3f}s"
    else:
        a, b = active_bounds(y)
        flux = spectral_flux(y[a:b])
        min_f = int(0.040 * SR / HOP)
        thr = flux.mean() + 1.5 * flux.std()
        peaks = [i for i in range(min_f, len(flux) - 1)
                 if flux[i] > thr and flux[i] >= flux[i - 1] and flux[i] >= flux[i + 1]]
        end = a + (peaks[0] * HOP if peaks else (b - a) // 3)
        seg, how = y[a:end], f"auto onset-to-flux-peak {a / SR:.3f}-{end / SR:.3f}s"
    core = seg[int(0.2 * len(seg)):int(0.8 * len(seg))]
    rate = (len(core) / SR) / (CONT_TARGET_MS / 1000)   # <1 lengthens
    out = librosa.effects.time_stretch(core, rate=rate) if len(core) > 512 else core
    return out, {"method": "ai-word-extraction", "keyword": word, "cut": how,
                 "stretch_rate": round(float(rate), 3)}

PHONEME_LOG = {}
for letter in CONTINUOUS:
    ipa, _, word, _ = TABLE4[letter]
    print(f"/{ipa}/ ({letter})")
    takes = []
    for rep in REPEATS:
        for spd in SPEEDS:
            ps = ipa * rep
            y, api = synth_phonemes(ps, VOICE, spd)
            if len(y) == 0:
                continue
            sf.write(os.path.join(OUT, "review", f"take_{letter}_r{rep}_s{spd}.wav"), y, SR, subtype="PCM_16")
            seg, flux = steadiest_window(y)
            ms = 1000.0 * len(seg) / SR
            score = flux + abs(ms - CONT_TARGET_MS) / 1000.0
            takes.append({"seg": seg, "ms": ms, "flux": flux, "score": score,
                          "settings": {"phonemes": ps, "repeat": rep, "speed": spd, "api": api}})
    passing = [t for t in takes if in_band(t["ms"])]
    path = os.path.join(OUT, "phonemes", f"ph_{letter}.wav")
    if passing:
        best = min(passing, key=lambda t: t["score"])
        finish_clip(best["seg"], path, already_trimmed=True)
        source, settings = "kokoro-phoneme-input", dict(best["settings"], steadiness_flux=round(best["flux"], 4))
    else:
        seg, settings = extract_from_keyword(letter, word)
        finish_clip(seg, path, already_trimmed=True)
        source = "ai-word-extraction"
    record(f"ph_{letter}", "phoneme", path, letter=letter, ipa=ipa, source=source,
           settings=dict(settings, candidates_tried=len(takes)))
    PHONEME_LOG[letter] = source
    print(f"  -> {source}, sound {MANIFEST['ph_' + letter]['soundMs']} ms")
    display(Audio(path))

print({s: sum(v == s for v in PHONEME_LOG.values()) for s in set(PHONEME_LOG.values())})

## 6. Short sounds: human recordings (optional)

Upload recordings named `ph_b.wav`, `ph_c.wav`, and so on (any sample rate, mono or stereo). They are resampled to 24 kHz and get the same editing standard. They are not voice-converted, because Kokoro has no speech-to-speech conversion. Short-sound letters without an upload are listed as `missing` in the manifest. Skip this cell if you have no recordings yet.

In [ ]:
from google.colab import files

SHORT = [l for l, v in TABLE4.items() if v[1] == "short"]
RECORDER = "human model"     # e.g. "Teacher A, coached"; recorded in the manifest

uploaded = files.upload()
for name, data in uploaded.items():
    stem = os.path.splitext(name)[0].lower()
    letter = stem.replace("ph_", "")
    if letter not in SHORT:
        print("skipped (not a short-sound letter):", name)
        continue
    tmp = os.path.join("/tmp", name)
    open(tmp, "wb").write(data)
    y, _ = librosa.load(tmp, sr=SR, mono=True)
    path = os.path.join(OUT, "phonemes", f"ph_{letter}.wav")
    finish_clip(y, path)
    record(f"ph_{letter}", "phoneme", path, letter=letter, ipa=TABLE4[letter][0],
           source="human-model", tool=RECORDER, voice=RECORDER, license_="own recording",
           settings={"original_file": name})
    print("added", f"ph_{letter}", MANIFEST[f"ph_{letter}"]["soundMs"], "ms")

## 7. Gate 1 (automated screen) and `manifest.json`

- **Length:** the sound without its pads, compared with Table 4. Continuous sounds must fall inside `CONT_BAND_MS` (an audit assumption, since Table 4 says only "about 800 ms"). Short sounds must be at or under their **[proposed]** limit.
- **Energy after release** (b c d g h j k p t x): the burst is the largest 5 ms RMS rise. The flag `possible_vowel_after_release` is raised if, 40 ms or more after the burst, there are at least 40 ms of voiced, periodic energy (autocorrelation peak in the 80–400 Hz pitch range above `VOICING_R`) within 25 dB of the clip peak. This catches "buh" and "tuh". j (/dʒ/) is voiced by nature and can raise false flags.
- q, w, and y include a glide that is voiced by nature, so they get the length check only.

`result` is `pass` or `fail` for length. Flags are advisory, and a flagged clip still needs the Praat check and the listening screen.

In [ ]:
RELEASE_CHECK = set("bcdghjkptx")
VOICING_R = 0.5
POST_RELEASE_MS = 40
MIN_VOICED_MS = 40

def voiced_after_release(y):
    y = trim_silence(y)
    db = frame_rms_db(y)
    if len(db) < 4:
        return False, None
    burst = int(np.argmax(np.diff(db)))
    start = burst * HOP + int(POST_RELEASE_MS / 1000 * SR)
    fr_len, lo, hi = int(0.030 * SR), int(SR / 400), int(SR / 80)
    voiced, pos, peak_db = 0, start, db.max()
    while pos + fr_len <= len(y):
        f = y[pos:pos + fr_len] - y[pos:pos + fr_len].mean()
        e = np.dot(f, f)
        if e > 0 and 10 * np.log10(e / fr_len + 1e-12) > peak_db - 25:
            ac = np.correlate(f, f, "full")[fr_len - 1:] / e
            if ac[lo:hi].max() > VOICING_R:
                voiced += HOP
        pos += HOP
    voiced_ms = 1000.0 * voiced / SR
    return voiced_ms >= MIN_VOICED_MS, round(voiced_ms)

rows = []
for letter, (ipa, kind, word, limit) in TABLE4.items():
    cid = f"ph_{letter}"
    if cid not in MANIFEST:
        MANIFEST[cid] = {"clipId": cid, "type": "phoneme", "letter": letter, "ipa": ipa,
                         "file": None, "source": "missing", "gate1": {"result": "missing"},
                         "gate2": "pending", "gate3": "pending", "version": CLIP_VERSION,
                         "releaseDate": None}
        rows.append((cid, kind, "-", "-", "missing", ""))
        continue
    e = MANIFEST[cid]
    y, _ = sf.read(os.path.join(OUT, e["file"]), dtype="float32")
    snd = measure_sound_ms(y)
    if kind == "continuous":
        target = f"{CONT_BAND_MS[0]}-{CONT_BAND_MS[1]} ms (audit band around ~{CONT_TARGET_MS})"
        ok = in_band(snd)
    else:
        target = f"<= {limit} ms [proposed]"
        ok = snd <= limit
    flags = []
    if kind == "short" and letter in RELEASE_CHECK:
        flagged, vms = voiced_after_release(y)
        if flagged:
            flags.append(f"possible_vowel_after_release ({vms} ms voiced)")
    elif kind == "short":
        flags.append("glide: release check not applied, listen")
    if e["source"] == "ai-word-extraction":
        flags.append("fallback extraction: check the cut by ear")
    e["gate1"] = {"result": "pass" if ok else "fail", "soundMs": round(snd), "target": target,
                  "flags": flags, "automated": True,
                  "note": "Automated screen only; Praat spectrogram check still required"}
    rows.append((cid, kind, round(snd), target, e["gate1"]["result"], "; ".join(flags)))

for r in rows:
    print("{:<6} {:<10} {:>6}  {:<42} {:<8} {}".format(*map(str, r)))

manifest = {
    "generated": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "tool": TOOL, "voice": VOICE, "license": LICENSE, "sampleRate": SR,
    "editing": {"padMs": PAD_MS, "fadeMs": FADE_MS, "targetRmsDbfs": TARGET_RMS_DBFS,
                "peakCeilingDbfs": PEAK_CEILING_DBFS, "format": "mono WAV PCM16"},
    "gate1Assumptions": {"continuousBandMs": list(CONT_BAND_MS), "shortLimitsProposed": True},
    "compose": {"lines": COMPOSE, "articulationCues": ARTICULATION_CUES,
                "substitutionCues": SUBSTITUTION_CUES},
    "clips": list(MANIFEST.values()),
}
with open(os.path.join(OUT, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(len(manifest["clips"]), "manifest entries written")

## 8. Zip and download

In [ ]:
from google.colab import files

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(f"/content/playit_audio_{stamp}", "zip", OUT)
print(archive, round(os.path.getsize(archive) / 1e6, 1), "MB")
files.download(archive)